<a href="https://colab.research.google.com/github/alyssanicoletech-star/Collab-Notebooks-/blob/main/Small_Language_Model_TinyStories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Small Language Model - Corrected Version (Causal LM, TinyStories)
# Author: Alyssa Nicole
# Date: June 2026
#
# Requires: pip install transformers datasets torch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset

# ============================
# CONFIGURATION (Colab GPU)
# ============================
class Config:
    vocab_size = 50257
    max_length = 256
    batch_size = 16
    embedding_dim = 512
    num_heads = 8
    num_layers = 8
    dropout = 0.1
    learning_rate = 2e-4
    epochs = 3
    num_train_examples = 20000   # subset of TinyStories for reasonable epoch time
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
print(f"Using device: {config.device}")

# ============================
# TOKENIZER
# ============================
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# ============================
# TRAINING DATA (TinyStories)
# ============================
print("Loading TinyStories dataset...")
tinystories = load_dataset("roneneldan/TinyStories", split="train")
sample_texts = tinystories["text"][:config.num_train_examples]
print(f"Loaded {len(sample_texts):,} training examples")

# ============================
# DATASET (shifted inputs/targets)
# ============================
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokenize with one extra slot so we can shift by one
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length + 1,
            padding='max_length',
            return_tensors='pt'
        )
        ids = encoded.input_ids.squeeze(0)

        # input = tokens[0:max_length], target = tokens[1:max_length+1]
        input_ids = ids[:-1]
        target_ids = ids[1:]
        return input_ids, target_ids

# ============================
# MODEL (causal self-attention)
# ============================
class SmallLanguageModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        self.pos_embedding = nn.Parameter(torch.zeros(1, config.max_length, config.embedding_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.embedding_dim,
            nhead=config.num_heads,
            dropout=config.dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)
        self.fc_out = nn.Linear(config.embedding_dim, config.vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        x = self.embedding(x) + self.pos_embedding[:, :seq_len, :]

        # Causal mask: position i cannot attend to positions > i
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)

        x = self.transformer(x, mask=causal_mask, is_causal=True)
        logits = self.fc_out(x)
        return logits

# ============================
# TRAINING FUNCTION
# ============================
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, outputs.size(-1)), targets.reshape(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# ============================
# TEXT GENERATION
# ============================
@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(config.device)

    for _ in range(max_new_tokens):
        # Keep within the model's positional embedding range
        if input_ids.size(1) > config.max_length:
            input_ids = input_ids[:, -config.max_length:]

        outputs = model(input_ids)
        next_token_logits = outputs[:, -1, :] / temperature
        next_token = torch.multinomial(torch.softmax(next_token_logits, dim=-1), num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

# ============================
# SETUP
# ============================
dataset = TextDataset(sample_texts, tokenizer, config.max_length)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)

model = SmallLanguageModel(config).to(config.device)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)

print(f"Setup complete! Model has {sum(p.numel() for p in model.parameters()):,} parameters")

# ============================
# TRAINING LOOP
# ============================
print("\nStarting training...\n")
for epoch in range(config.epochs):
    print(f"Epoch {epoch+1}/{config.epochs}")
    avg_loss = train_epoch(model, dataloader, optimizer, criterion, config.device)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}\n")

print("Training finished!")

# Test generation
prompt = "Once upon a time"
generated = generate_text(model, tokenizer, prompt, max_new_tokens=80)
print(f"\nPrompt: {prompt}")
print(f"Generated: {generated}")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Loaded 20,000 training examples


/tmp/ipykernel_4182/3186100355.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config.num_layers)


Setup complete! Model has 76,863,569 parameters

Starting training...

Epoch 1/3
  Batch 0/1250 | Loss: 11.8706
  Batch 10/1250 | Loss: 7.2419
  Batch 20/1250 | Loss: 5.7228
  Batch 30/1250 | Loss: 5.2371
  Batch 40/1250 | Loss: 5.0490
  Batch 50/1250 | Loss: 4.5805
  Batch 60/1250 | Loss: 4.4705
  Batch 70/1250 | Loss: 4.2810
  Batch 80/1250 | Loss: 4.5433
  Batch 90/1250 | Loss: 4.2623
  Batch 100/1250 | Loss: 4.1925
  Batch 110/1250 | Loss: 4.0441
  Batch 120/1250 | Loss: 4.2691
  Batch 130/1250 | Loss: 4.1205
  Batch 140/1250 | Loss: 3.9055
  Batch 150/1250 | Loss: 3.8286
  Batch 160/1250 | Loss: 3.8945
  Batch 170/1250 | Loss: 3.9761
  Batch 180/1250 | Loss: 3.8036
  Batch 190/1250 | Loss: 3.7703
  Batch 200/1250 | Loss: 3.8326
  Batch 210/1250 | Loss: 3.8317
  Batch 220/1250 | Loss: 3.9067
  Batch 230/1250 | Loss: 3.6834
  Batch 240/1250 | Loss: 3.7608
  Batch 250/1250 | Loss: 3.5797
  Batch 260/1250 | Loss: 3.7659
  Batch 270/1250 | Loss: 3.8761
  Batch 280/1250 | Loss: 3.6390
 